# Brain Tumor MRI Classification - Data Exploration

This notebook provides an exploration of the Brain Tumor MRI Dataset.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Import custom modules
from data_utils import BrainTumorDataLoader, get_data_info

## 1. Dataset Information

In [ ]:
# Display dataset information
get_data_info('../data')

## 2. Load Dataset

In [ ]:
# Initialize data loader
data_loader = BrainTumorDataLoader(data_dir='../data', img_size=(224, 224))

# Load dataset
X_train, X_val, X_test, y_train, y_val, y_test = data_loader.load_dataset(
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

print(f"\nDataset shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

## 3. Class Distribution

In [ ]:
# Get class distributions
train_dist = data_loader.get_class_distribution(y_train)
val_dist = data_loader.get_class_distribution(y_val)
test_dist = data_loader.get_class_distribution(y_test)

# Create a DataFrame
dist_df = pd.DataFrame({
    'Training': train_dist,
    'Validation': val_dist,
    'Testing': test_dist
})

print("Class Distribution:")
print(dist_df)
print(f"\nTotal: {dist_df.sum(axis=0)}")

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Training set
axes[0].bar(train_dist.keys(), train_dist.values())
axes[0].set_title('Training Set Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=45)

# Validation set
axes[1].bar(val_dist.keys(), val_dist.values())
axes[1].set_title('Validation Set Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Images')
axes[1].tick_params(axis='x', rotation=45)

# Test set
axes[2].bar(test_dist.keys(), test_dist.values())
axes[2].set_title('Test Set Distribution')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Number of Images')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Sample Images Visualization

In [ ]:
# Visualize sample images from each class
num_classes = len(data_loader.classes)
samples_per_class = 5

fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(15, 3*num_classes))

for class_idx in range(num_classes):
    # Find indices for this class
    class_indices = np.where(y_train == class_idx)[0]
    
    # Sample random images
    sample_indices = np.random.choice(class_indices, samples_per_class, replace=False)
    
    for i, idx in enumerate(sample_indices):
        ax = axes[class_idx, i] if num_classes > 1 else axes[i]
        ax.imshow(X_train[idx])
        ax.axis('off')
        if i == 0:
            ax.set_title(f"{data_loader.classes[class_idx]}", fontsize=12, fontweight='bold')

plt.suptitle('Sample MRI Images from Each Class', fontsize=16, fontweight='bold', y=1.001)
plt.tight_layout()
plt.show()

## 5. Image Statistics

In [ ]:
# Calculate statistics
print("Image Statistics:")
print(f"Mean pixel value: {X_train.mean():.4f}")
print(f"Std pixel value: {X_train.std():.4f}")
print(f"Min pixel value: {X_train.min():.4f}")
print(f"Max pixel value: {X_train.max():.4f}")

In [ ]:
# Plot pixel distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sample some images for analysis
sample_size = min(1000, len(X_train))
sample_indices = np.random.choice(len(X_train), sample_size, replace=False)
sample_images = X_train[sample_indices]

# Red channel
axes[0].hist(sample_images[:, :, :, 0].flatten(), bins=50, color='red', alpha=0.7)
axes[0].set_title('Red Channel Distribution')
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Frequency')

# Green channel
axes[1].hist(sample_images[:, :, :, 1].flatten(), bins=50, color='green', alpha=0.7)
axes[1].set_title('Green Channel Distribution')
axes[1].set_xlabel('Pixel Value')
axes[1].set_ylabel('Frequency')

# Blue channel
axes[2].hist(sample_images[:, :, :, 2].flatten(), bins=50, color='blue', alpha=0.7)
axes[2].set_title('Blue Channel Distribution')
axes[2].set_xlabel('Pixel Value')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Next Steps

Now that we've explored the data, we can proceed to:

1. Train a model using the training script:
   ```bash
   python src/train.py --model-type simple --epochs 50
   ```

2. Or experiment with different architectures and hyperparameters in this notebook

3. Evaluate the trained model on the test set

4. Make predictions on new images using:
   ```bash
   python src/predict.py --model models/model.h5 --image path/to/image.jpg
   ```